In [49]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os, pickle

import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'NanumGothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

In [50]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 함수 생성

In [51]:
def load_file(prep_fold):
  fn = os.listdir(path+prep_fold)

  for f in fn :
    if 'test' in f:
      test_n = f
    else :
      train_n = f

  train = pd.read_parquet(path + prep_fold +'/' + train_n)
  test =  pd.read_parquet(path + prep_fold +'/' + test_n)

  print(f'train file name : {train_n}')
  print(f'test file name : {test_n}')
  return train, test

In [52]:
def cols_prep(train_df, test_df) :
  # train columns drop
  drop_cols = ['기준년월','ID','Segment']
  x = train_df.drop(columns=drop_cols)
  y = train_df['Segment']

  # test 데이터 컬럼 train 데이터와 동일하게 맞추기
  X_test = test_df[x.columns]

  return x, y, X_test

In [53]:
def modeling(learn_rate, version,estimator) :
  model = lgb.LGBMClassifier(
      objective='multiclass',
      num_class=5,
      n_estimators=2000,
      learning_rate=learn_rate,
      random_state=42,
      n_jobs=-1,
      min_gain_to_split=1e-5,
      min_data_in_leaf=10,
      subsample=0.8,
      colsample_bytree=0.8
  )

  model.fit(
      x_train, y_train,
      eval_set=[(x_test, y_test)],
      eval_metric='multi_logloss',
      callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)]
  )

  # 모델 저장
  pickle.dump(model, open(f'/content/drive/MyDrive/Colab_Notebooks/LGBM_{version}_est_{estimator}_learning_rate{learn_rate}.pkl', 'wb'))

  return model

In [54]:
def evaluate_multiclass(y_true, y_pred, y_proba, learn_rate, version, estimator):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')

    # ROC AUC: 다중 분류는 one-vs-rest 방식 필요
    try:
        roc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except:
        roc = np.nan  # 예외 발생 시

    # G-Mean: 각 클래스의 recall 평균으로 근사
    cm = confusion_matrix(y_true, y_pred)
    recalls = []
    for i in range(len(cm)):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        if tp + fn != 0:
            recalls.append(tp / (tp + fn))
    gmean = np.sqrt(np.prod(recalls)) if all(r > 0 for r in recalls) else 0

    score_dict = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'ROC AUC': roc,
        'G-Mean': gmean
    }

    pd.DataFrame([score_dict]).to_csv(f'/content/drive/MyDrive/Colab_Notebooks/LGBM_{version}_est_{estimator}_score_learn_rate{learn_rate}.csv', index=False)

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"ROC AUC   : {roc:.4f}")
    print(f"G-Mean    : {gmean:.4f}")

In [55]:
def save_data(test, final_pred, learn_rate, version, estimator):
  res_df = pd.DataFrame({
      'ID': test.ID,
      'Segment': final_pred
  })

  res_df.to_csv(f'/content/drive/MyDrive/Colab_Notebooks/submission_raw_{version}_learning_rate_{learn_rate}.csv', index=False)

  submission_unique = res_df.groupby('ID')['Segment'].agg(lambda x: x.mode().iloc[0]).reset_index()
  submission_unique.to_csv(f'/content/drive/MyDrive/Colab_Notebooks/submission_{version}_est_{estimator}_learning_rate_{learn_rate}.csv', index=False)

In [56]:
# 데이터 경로
print(os.getcwd())
path = '/content/drive/MyDrive/파이널 프로젝트/2025_07_09/' #'/content/drive/MyDrive/Colab_Notebooks/01.like_lion_final_prj/'
prep_folder = os.listdir(path)

/content


In [57]:
prep_folder

['분산(0.01),열의상관(0.7이상),Segment의상관(0.1이상)',
 '분산(0.01),열의상관(0.8이상),Segment의상관(0.1이상)',
 '분산(0.01),열의상관(0.8이상),Segment의상관(0.1이상,열의 상관중 Segment와 0.3이상 상관이 있으면 열삭제 x)']

## 전처리 ver02

In [58]:
train, test = load_file(prep_folder[0])

train file name : Segment_merge_ver_02.parquet
test file name : Segment_merge_test_ver_02.parquet


In [59]:
print(train.shape)
print(test.shape)

(2400000, 66)
(600000, 65)


In [60]:
x, y, X_test = cols_prep(train, test)

In [61]:
 # 데이터 분할
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [65]:
# 모델링
## learning rate 0.1
# estimators [200, 500, 1000]

estimator = [200, 500, 1000]
for e in estimator:
  model = modeling(learn_rate=0.1, version='ver02', estimator=e)

  # 모델 평가 지표
  y_pred = model.predict(x_test)
  y_proba = model.predict_proba(x_test)  # shape: (n_samples, n_classes)

  evaluate_multiclass(y_test, y_pred, y_proba, learn_rate=0.1, version='ver02',estimator= e)

  # 예측
  final_pred = model.predict(X_test)
  final_proba = model.predict_proba(X_test)
  # 결과 저장
  save_data(test, final_pred, learn_rate=0.1, version='ver02', estimator=e )

  #### 실행 종료
  print(f'##########################estimator {e} : 실행종료##########################################')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.424180 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8996
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 63
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] mi

- learning rate = 0.05

In [67]:
# 모델링
## learning rate 0.05
# estimators [200, 500, 1000]

estimator = [200, 500, 1000]
for e in estimator:
  model = modeling(learn_rate=0.05, version='ver02', estimator=e)

  # 모델 평가 지표
  y_pred = model.predict(x_test)
  y_proba = model.predict_proba(x_test)  # shape: (n_samples, n_classes)

  evaluate_multiclass(y_test, y_pred, y_proba, learn_rate=0.05, version='ver02',estimator= e)

  # 예측
  final_pred = model.predict(X_test)
  final_proba = model.predict_proba(X_test)
  # 결과 저장
  save_data(test, final_pred, learn_rate=0.05, version='ver02', estimator=e )

  #### 실행 종료
  print(f'##########################estimator {e} : 실행종료##########################################')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.235958 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8996
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 63
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] mi

In [66]:
path

'/content/drive/MyDrive/파이널 프로젝트/2025_07_09/'

In [48]:
import filecmp
path = '/content/drive/MyDrive/Colab_Notebooks/'


file1 = path + 'submission_ver03_learning_rate_0.1.csv'
file2 = path + 'submission_ver05_learning_rate_0.05.csv'

if filecmp.cmp(file1, file2, shallow=False):
    print("두 파일은 동일합니다.")
else:
    print("두 파일은 다릅니다.")

두 파일은 다릅니다.


In [ ]:
## 같은 파일
'submission_ver02_learning_rate_0.1.csv'
'submission_ver03_learning_rate_0.1.csv'